In [ ]:
# VitalLens 실시간 스트리밍 예제 -> pos 기법으로 심박수만 탐지

import cv2
import time
import numpy as np
from IPython.display import display, Image, clear_output
from vitallens import VitalLens

METHOD  = "vitallens"
API_KEY = "pUhOjDlWwi5X5v8lfXy2i3mTOhu0fknw3RomkKZJ"
CAM_ID  = 0

cap = cv2.VideoCapture(CAM_ID)

if not cap.isOpened():
    print(f"❌ 에러: {CAM_ID}번 카메라를 열 수 없습니다.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    vl = VitalLens(method=METHOD, api_key=API_KEY)

    last_face_box = None
    hr_val  = None
    hr_conf = None
    rr_val  = None  # 호흡수(Respiratory Rate) 값 변수 추가
    rr_conf = None  # 호흡수 신뢰도 변수 추가

    print("✅ 실행 중... 중단하려면 Kernel > Interrupt")

    try:
        with vl.stream() as session:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                session.push(rgb, timestamp=time.time())

                results = session.get_result(block=False)
                if results and len(results) > 0:
                    face   = results[0]
                    coords = face.get("face", {}).get("coordinates")
                    if coords is not None and len(coords) > 0:
                        last_face_box = coords[-1]

                    vitals  = face.get("vitals", {})
                    
                    # 1. 심박수(Heart Rate) 추출
                    hr_data = vitals.get("heart_rate", {})
                    hr_val  = hr_data.get("value")
                    hr_conf = hr_data.get("confidence")

                    # 2. 호흡수(Respiratory Rate) 추출 추가
                    rr_data = vitals.get("respiratory_rate", {})
                    rr_val  = rr_data.get("value")
                    rr_conf = rr_data.get("confidence")

                # 얼굴 박스 그리기
                if last_face_box is not None:
                    x1, y1, x2, y2 = [int(v) for v in last_face_box]
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 128), 2)

                # 심박수 텍스트 출력
                if hr_val is not None:
                    hr_color = (0, 255, 128) if (hr_conf or 0) > 0.6 else (0, 200, 255)
                    cv2.putText(frame, f"HR: {hr_val:.1f} bpm", (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.8, hr_color, 2)
                    if hr_conf:
                        cv2.putText(frame, f"HR Conf: {hr_conf:.0%}", (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180,180,180), 1)
                else:
                    cv2.putText(frame, "Detecting...", (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (100, 200, 255), 2)

                # 호흡수 텍스트 출력 (심박수 아래에 배치)
                if rr_val is not None:
                    rr_color = (255, 150, 50) if (rr_conf or 0) > 0.6 else (200, 100, 50)
                    cv2.putText(frame, f"RR: {rr_val:.1f} rpm", (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.8, rr_color, 2)
                    if rr_conf:
                        cv2.putText(frame, f"RR Conf: {rr_conf:.0%}", (20, 135), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180,180,180), 1)

                _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
                clear_output(wait=True)
                display(Image(data=buf.tobytes()))

    except KeyboardInterrupt:
        print("🛑 사용자가 종료했습니다.")
    finally:
        cap.release()

In [ ]:
# VitalLens 실시간 스트리밍 예제 -> 호흡수, 심박수 다 탐지

import cv2
import time
import numpy as np
from IPython.display import display, Image, clear_output
from vitallens import VitalLens

METHOD  = "pos"
CAM_ID  = 0

cap = cv2.VideoCapture(CAM_ID)

if not cap.isOpened():
    print(f"❌ 에러: {CAM_ID}번 카메라를 열 수 없습니다.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    vl = VitalLens(method=METHOD)

    # 💡 [핵심 해결책] 라이브러리 버그 우회를 위한 몽키 패치!
    # pos 모드에 없는 input_size 속성을 강제로 만들어줍니다.
    if not hasattr(vl.rppg, 'input_size'):
        vl.rppg.input_size = 224  

    last_face_box = None
    hr_val  = None
    hr_conf = None
    rr_val  = None  
    rr_conf = None  

    print(f"✅ [{METHOD}] 모드로 실행 중... 중단하려면 Kernel > Interrupt")

    try:
        with vl.stream() as session:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                session.push(rgb, timestamp=time.time())

                results = session.get_result(block=False)
                if results and len(results) > 0:
                    face = results[0]
                    
                    face_data = face.get("face") or {}
                    coords = face_data.get("coordinates")
                    if coords is not None and len(coords) > 0:
                        last_face_box = coords[-1]

                    vitals = face.get("vitals") or {}
                    
                    # 1. 심박수(Heart Rate) 추출
                    hr_data = vitals.get("heart_rate") or {}
                    hr_val  = hr_data.get("value")
                    hr_conf = hr_data.get("confidence")

                    # 2. 호흡수(Respiratory Rate) 추출 
                    rr_data = vitals.get("respiratory_rate") or {}
                    rr_val  = rr_data.get("value")
                    rr_conf = rr_data.get("confidence")

                # 얼굴 박스 그리기
                if last_face_box is not None:
                    x1, y1, x2, y2 = [int(v) for v in last_face_box]
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 128), 2)

                # 심박수 텍스트 출력
                if hr_val is not None:
                    hr_color = (0, 255, 128) if (hr_conf or 0) > 0.6 else (0, 200, 255)
                    cv2.putText(frame, f"HR: {hr_val:.1f} bpm", (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.8, hr_color, 2)
                else:
                    cv2.putText(frame, "Detecting...", (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (100, 200, 255), 2)

                # 호흡수 텍스트 출력
                if rr_val is not None:
                    rr_color = (255, 150, 50) if (rr_conf or 0) > 0.6 else (200, 100, 50)
                    cv2.putText(frame, f"RR: {rr_val:.1f} rpm", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, rr_color, 2)

                _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
                clear_output(wait=True)
                display(Image(data=buf.tobytes()))

    except KeyboardInterrupt:
        print("🛑 사용자가 종료했습니다.")
    finally:
        cap.release()